In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Load datasets
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
mangroves_FN_path = base_path / "Mangroves_FN" / "mangroves.shp"
seagrass_path = base_path / "seagrass_clipped_10000m.shp"
corals_path = base_path / "corals_clipped_1000m.shp"

In [ ]:
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)

In [ ]:
# Step 2: Ensure the Land Use Layer is in Jamaica's CRS
jam_crs = jamaica_boundary.crs
print("Jamaica boundary CRS:", jam_crs)

In [ ]:
# Load and reproject datasets
mangroves_FN = gpd.read_file(mangroves_FN_path).to_crs(jam_crs)
seagrass = gpd.read_file(seagrass_path).to_crs(jam_crs)
corals = gpd.read_file(corals_path).to_crs(jam_crs)

In [ ]:
# Buffer zones
buffers = [250, 500, 1000]  # Buffer distances in meters
mangroves_FN_buffers = [mangroves_FN.buffer(distance) for distance in buffers]
coral_buffers = [corals.buffer(distance) for distance in buffers]
seagrass_buffers = [seagrass.buffer(distance) for distance in buffers]

In [ ]:
# # Example: 500m Mangroves to Coral
# mangrove_500m = mangroves_FN_buffers[1]
# # Extract geometries where mangroves are within 500m of corals
# mangroves_near_corals = mangroves_FN[mangrove_500m.intersects(corals.unary_union)]

In [ ]:
# Overlap of all three ecosystems
all_ecosystems_overlap = (
    mangroves_FN.geometry.intersection(corals.geometry.unary_union)
    .intersection(seagrass.geometry.unary_union)
)

In [ ]:
# # Plotting
# fig, ax = plt.subplots(figsize=(12, 12))

# # Plot base layers
# jamaica_boundary.plot(ax=ax, color="lightgrey", edgecolor="black")
# mangroves_FN.plot(ax=ax, color="green", alpha=0.5)
# corals.plot(ax=ax, color="blue", alpha=0.5)
# seagrass.plot(ax=ax, color="yellow", alpha=0.5)

# # Plot derived layers
# if not mangroves_near_corals.empty:
#     mangroves_near_corals.plot(ax=ax, color="red", alpha=0.5)
# if not all_ecosystems_overlap.empty:
#     gpd.GeoSeries(all_ecosystems_overlap).plot(ax=ax, color="purple", alpha=0.7)

# # Create manual legend patches
# legend_patches = [
#     mpatches.Patch(color="lightgrey", label="Jamaica Boundary"),
#     mpatches.Patch(color="green", label="Mangroves"),
#     mpatches.Patch(color="blue", label="Corals"),
#     mpatches.Patch(color="yellow", label="Seagrass"),
#     mpatches.Patch(color="red", label="Mangroves near Corals (500m)"),
#     mpatches.Patch(color="purple", label="Overlap of All Ecosystems"),
# ]

# # Add manual legend
# ax.legend(handles=legend_patches, loc="upper left", fontsize=10, title="Legend", title_fontsize=12)

# # Add title and axis labels
# ax.set_title("Spatial Analysis of Ecosystems in Jamaica", fontsize=16)
# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")

# plt.show()

In [ ]:
# Create dictionary to store overlaps for each buffer distance
overlaps = {}

for distance in buffers:
    # Create buffers for each dataset at the given distance
    mangrove_buffer = mangroves_FN.buffer(distance)
    coral_buffer = corals.buffer(distance)
    seagrass_buffer = seagrass.buffer(distance)
    
    # Compute the intersection (overlap) of all three ecosystems at this buffer distance
    intersection = (mangrove_buffer.unary_union
                    .intersection(coral_buffer.unary_union)
                    .intersection(seagrass_buffer.unary_union))
    
    # Save the result if it's not empty
    if not intersection.is_empty:
        # intersection could be a GeometryCollection, so wrap it in GeoSeries
        overlaps[distance] = gpd.GeoSeries([intersection], crs=jam_crs)


# Define colors for each buffer distance
buffer_colors = {
    250: "orange",
    500: "red",
    1000: "purple"
}

# Initialize plot
fig, ax = plt.subplots(figsize=(12, 12))

# Plot the Jamaica boundary as a base layer
jamaica_boundary.plot(ax=ax, color="none", edgecolor="black", linewidth=1, label="Jamaica Boundary")

# Plot overlaps for each buffer distance with distinct colors
for distance, overlap in overlaps.items():
    overlap.plot(ax=ax, color=buffer_colors[distance], alpha=0.5, label=f"Overlap @ {distance}m")

# Create manual legend patches for boundaries and overlaps
legend_patches = [
    mpatches.Patch(facecolor="none", edgecolor="black", label="Jamaica Boundary"),
]
for distance, color in buffer_colors.items():
    legend_patches.append(mpatches.Patch(color=color, alpha=0.5, label=f"Overlap @ {distance}m"))

# Add manual legend to the plot
ax.legend(handles=legend_patches, loc="upper left", fontsize=10, title="Legend", title_fontsize=12)

# Add title and axis labels
ax.set_title("Overlap of All Ecosystems at Different Buffers in Jamaica", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

In [ ]:
# Define buffer distances
buffers = [250, 500, 1000]

# Dictionary to store exclusive overlaps for each buffer
exclusive_overlaps = {}

for distance in buffers:
    # Create buffers for each dataset at the given distance
    mangrove_buffer = mangroves_FN.buffer(distance)
    coral_buffer = corals.buffer(distance)
    seagrass_buffer = seagrass.buffer(distance)
    
    # Compute intersections for mangroves with corals and seagrass
    mangrove_coral = mangrove_buffer.intersection(coral_buffer)
    mangrove_seagrass = mangrove_buffer.intersection(seagrass_buffer)
    
    # Compute exclusive overlaps:
    # Areas where mangroves & corals overlap but not seagrass
    exclusive_m_c = mangrove_coral.difference(seagrass_buffer.unary_union)
    # Areas where mangroves & seagrass overlap but not corals
    exclusive_m_s = mangrove_seagrass.difference(coral_buffer.unary_union)
    
    # Use unary_union to merge all resulting geometries into a single geometry
    union_geom = exclusive_m_c.unary_union.union(exclusive_m_s.unary_union)
    
    # Check if the resulting geometry is not empty
    if not union_geom.is_empty:
        # Wrap the unified geometry in a GeoSeries for plotting
        exclusive_overlaps[distance] = gpd.GeoSeries([union_geom], crs=jam_crs)

# Define colors for each buffer distance
buffer_colors = {
    250: "cyan",
    500: "magenta",
    1000: "yellow"
}

# Plotting
fig, ax = plt.subplots(figsize=(12, 12))

# Plot Jamaica boundary as context
jamaica_boundary.plot(ax=ax, color="none", edgecolor="black", linewidth=1)

# Plot exclusive overlaps for each buffer distance
for distance, excl_overlap in exclusive_overlaps.items():
    excl_overlap.plot(ax=ax, color=buffer_colors[distance], alpha=0.5, label=f"Exclusive Overlap @ {distance}m")

# Create manual legend patches
legend_patches = [mpatches.Patch(facecolor="none", edgecolor="black", label="Jamaica Boundary")]
for distance, color in buffer_colors.items():
    if distance in exclusive_overlaps:
        legend_patches.append(mpatches.Patch(color=color, alpha=0.5, label=f"Exclusive Overlap @ {distance}m"))

# Add legend to the plot
ax.legend(handles=legend_patches, loc="upper left", fontsize=10, title="Legend", title_fontsize=12)

# Add title and axis labels
ax.set_title("Exclusive Overlap of Mangroves with Either Seagrass or Corals (Not Both)", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()